In [27]:
import numpy as np
import pandas as pd
import seaborn as sns

class display(object):
    """Display HTML representation of multiple objects"""
    template = """<div style="float: left; padding: 10px;">
    <p style='font-family:"Courier New", Courier, monospace'>{0}</p>{1}
    </div>"""
    def __init__(self, *args):
        self.args = args
        
    def _format_html_value(self, obj):
        html_method = getattr(obj, '_repr_html_', None)
        if callable(html_method):
            return html_method()
        # Fallback for scalars or objects without HTML repr
        return f"<pre>{repr(obj)}</pre>"
    
    def _repr_html_(self):
        return '\n'.join(
            self.template.format(a, self._format_html_value(eval(a)))
            for a in self.args
        )
    
    def __repr__(self):
        return '\n\n'.join(a + '\n' + repr(eval(a))
                           for a in self.args)

In [28]:
taxis = sns.load_dataset('taxis')
taxis_numeric = taxis.select_dtypes(include=np.number)
print('taxis: ', taxis.info())

<class 'pandas.DataFrame'>
RangeIndex: 6433 entries, 0 to 6432
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   pickup           6433 non-null   datetime64[us]
 1   dropoff          6433 non-null   datetime64[us]
 2   passengers       6433 non-null   int64         
 3   distance         6433 non-null   float64       
 4   fare             6433 non-null   float64       
 5   tip              6433 non-null   float64       
 6   tolls            6433 non-null   float64       
 7   total            6433 non-null   float64       
 8   color            6433 non-null   str           
 9   payment          6389 non-null   str           
 10  pickup_zone      6407 non-null   str           
 11  dropoff_zone     6388 non-null   str           
 12  pickup_borough   6407 non-null   str           
 13  dropoff_borough  6388 non-null   str           
dtypes: datetime64[us](2), float64(5), int64(1), str(6)


# Simple aggragation

In [29]:
series = pd.Series(range(99))
display('series.mean()', 'taxis_numeric.mean()', 'taxis_numeric.mean(axis=1)')

series.mean()
np.float64(49.0)

taxis_numeric.mean()
passengers     1.539251
distance       3.024617
fare          13.091073
tip            1.979220
tolls          0.325273
total         18.517794
dtype: float64

taxis_numeric.mean(axis=1)
0        4.116667
1        2.681667
2        4.398333
3       13.133333
4        4.776667
          ...    
6428     2.278333
6429    22.756667
6430     6.406667
6431     2.486667
6432     7.228333
Length: 6433, dtype: float64

Summarizing various built-in aggregatins.

In [30]:
taxis.describe()

,pickup,dropoff,passengers,distance,fare,tip,tolls,total
count,6433,6433,6433.000000,6433.000000,6433.000000,6433.00000,6433.000000,6433.000000
mean,2019-03-16 08:31:28.514223,2019-03-16 08:45:49.491216,1.539251,3.024617,13.091073,1.97922,0.325273,18.517794
min,2019-02-28 23:29:03,2019-02-28 23:32:35,0.000000,0.000000,1.000000,0.00000,0.000000,1.300000
25%,2019-03-08 15:50:34,2019-03-08 16:12:51,1.000000,0.980000,6.500000,0.00000,0.000000,10.800000
50%,2019-03-15 21:46:58,2019-03-15 22:06:44,1.000000,1.640000,9.500000,1.70000,0.000000,14.160000
75%,2019-03-23 17:41:38,2019-03-23 17:51:56,2.000000,3.210000,15.000000,2.80000,0.000000,20.300000
max,2019-03-31 23:43:45,2019-04-01 00:13:58,6.000000,36.700000,150.000000,33.20000,24.020000,174.820000
std,NaN,NaN,1.203768,3.827867,11.551804,2.44856,1.415267,13.815570


# groupby: split, apply, combine

In [40]:
display('taxis_numeric.groupby(\'passengers\').sum()', 'taxis.groupby(\'payment\')[\'distance\'].mean()')

,distance,fare,tip,tolls,total
passengers,,,,,
0,284.20,1222.50,229.97,51.84,1820.81
1,14124.03,60987.37,8992.47,1492.58,85736.77
2,2609.25,11534.50,1843.40,281.50,16563.85
3,762.01,3406.00,538.30,107.30,4874.65
4,323.01,1434.50,250.82,34.56,2096.63
5,821.08,3481.50,563.37,74.88,5025.05
6,533.78,2148.50,313.99,49.82,3007.21


### dispatch methods

In [44]:
taxis.groupby('passengers')['fare'].describe().unstack()

       passengers
count  0               96.000000
       1             4678.000000
       2              876.000000
       3              243.000000
       4              110.000000
       5              277.000000
       6              153.000000
mean   0               12.734375
       1               13.037061
       2               13.167237
       3               14.016461
       4               13.040909
       5               12.568592
       6               14.042484
std    0               10.116155
       1               11.168464
       2               12.765411
       3               14.589342
       4                9.963035
       5                9.658048
       6               14.976731
min    0                2.500000
       1                1.000000
       2                2.500000
       3                3.000000
       4                3.000000
       5                3.500000
       6                2.500000
25%    0                6.500000
       1                6

## aggregate, filter, transform, apply

### aggregate

In [55]:
print(taxis_numeric.groupby('passengers')[['fare', 'distance']].aggregate(['min', 'mean', np.max]))
print(taxis_numeric.groupby('passengers')[['fare', 'distance']].aggregate({'fare': 'min', 'distance': 'mean'}))

           fare                   distance                 
            min       mean    max      min      mean    max
passengers                                                 
0           2.5  12.734375   52.0     0.00  2.960417  21.10
1           1.0  13.037061  130.0     0.00  3.019245  36.66
2           2.5  13.167237  150.0     0.00  2.978596  36.70
3           3.0  14.016461  120.0     0.00  3.135844  19.62
4           3.0  13.040909   52.0     0.20  2.936455  19.16
5           3.5  12.568592   52.0     0.28  2.964188  21.51
6           2.5  14.042484  143.5     0.00  3.488758  33.76
            fare  distance
passengers                
0            2.5  2.960417
1            1.0  3.019245
2            2.5  2.978596
3            3.0  3.135844
4            3.0  2.936455
5            3.5  2.964188
6            2.5  3.488758


### filter
gets rows what the mean total is less than 19 for each group of passenger counts.

In [110]:
print(taxis.groupby('passengers').filter(lambda x: x['total'].mean() < 19).shape)
print(taxis.shape)

(5927, 14)
(6433, 14)


### transformations

In [118]:
print(taxis_numeric.groupby('passengers').transform(lambda x: round(x)).head())
print(taxis_numeric.groupby('passengers').head())

   distance  fare  tip  tolls  total
0       2.0   7.0  2.0    0.0   13.0
1       1.0   5.0  0.0    0.0    9.0
2       1.0   8.0  2.0    0.0   14.0
3       8.0  27.0  6.0    0.0   37.0
4       2.0   9.0  1.0    0.0   13.0
     passengers  distance  fare   tip  tolls  total
0             1      1.60   7.0  2.15   0.00  12.95
1             1      0.79   5.0  0.00   0.00   9.30
2             1      1.37   7.5  2.36   0.00  14.16
3             1      7.70  27.0  6.15   0.00  36.95
4             3      2.16   9.0  1.10   0.00  13.40
5             1      0.49   7.5  2.16   0.00  12.96
7             0      1.40   8.5  0.00   0.00  11.80
14            0      2.90  11.5  0.00   0.00  14.80
15            3      2.09  13.5  0.00   0.00  16.80
19            6      1.08   6.5  1.08   0.00  11.38
23            5      1.09   8.5  2.36   0.00  14.16
24            5      2.89   9.5  3.84   0.00  16.64
31            3      0.74   4.0  0.00   0.00   7.80
33            2      0.65   4.5  2.49   0.00  10.7

### apply method

In [120]:
taxis_numeric.groupby('passengers').apply(lambda x : x['fare'])

passengers      
0           7        8.5
            14      11.5
            41      20.0
            92      33.5
            117     15.5
                    ... 
6           6015     9.5
            6155     6.5
            6350     8.5
            6369    23.0
            6421    33.0
Name: fare, Length: 6433, dtype: float64

## specifying the split key

In [126]:
small_df = taxis_numeric.iloc[:10, :10]
indices = [0, 1, 3, 1, 3, 0, 2, 0, 1, 3]
display('small_df', 'small_df.groupby(indices).sum()')

,passengers,distance,fare,tip,tolls,total
0,1,1.60,7.0,2.15,0.0,12.95
1,1,0.79,5.0,0.00,0.0,9.30
2,1,1.37,7.5,2.36,0.0,14.16
3,1,7.70,27.0,6.15,0.0,36.95
4,3,2.16,9.0,1.10,0.0,13.40
5,1,0.49,7.5,2.16,0.0,12.96
6,1,3.65,13.0,2.00,0.0,18.80
7,0,1.40,8.5,0.00,0.0,11.80
8,1,3.63,15.0,1.00,0.0,19.30
9,1,1.52,8.0,1.00,0.0,13.30


### dictionary or series mapping index to group

In [129]:
taxis_numeric.groupby({0:1}).sum()

,passengers,distance,fare,tip,tolls,total
1.0,1,1.6,7.0,2.15,0.0,12.95


### any python func

In [132]:
taxis_numeric.groupby(lambda x : x % 5).mean()

,passengers,distance,fare,tip,tolls,total
0,1.574204,2.951950,12.856977,1.970995,0.325641,18.257148
1,1.575758,3.044810,13.172370,2.009611,0.323768,18.645726
2,1.521368,3.006861,13.127669,1.975991,0.329526,18.549891
3,1.475894,3.004137,13.042636,1.896384,0.318974,18.367208
4,1.548989,3.115381,13.255801,2.043103,0.328453,18.769075
